In [ ]:
import sys
import os
from google.colab import drive


In [ ]:
import os

outcar_path = "OUTCAR"

if os.path.exists(outcar_path):
#dimension
    file_size_mb = os.path.getsize(outcar_path) / (1024 * 1024)
    print(f"Dimensione OUTCAR: {file_size_mb:.2f} MB")

#number structure
    with open(outcar_path, 'r', encoding='latin-1', errors='ignore') as f:
        n_steps = sum(1 for line in f if "POSITION" in line)

    print(f" Number structure/steps: {n_steps}")
else:
    print("File OUTCAR not found.")

In [ ]:
import os
import numpy as np
from ase.io import read, write
from google.colab import drive

drive.mount('/content/drive')

outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/repulsive"
os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::1", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")


# FIX ASE KEYS
print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

# Empirical repulsive prior

As experienced with the MACE toy model, one property that MACE sometimes struggle the most to learn is to avoid the overlap of the atoms and so to learn the repulsive force between nuclei. With lower computational resources it's possible to implant a prior knowledge of a repulsive potential in the MACE learning. MACE can automatically apply a physical repulsive core (typically the Ziegler-Biersack-Littmark, or ZBL, potential) at short distances. The neural network then only has to learn the difference between the real DFT energy and the baseline repulsive energy.

Training script

In [ ]:
# hyperpars for MACE training (sources: MACE T01 tutorial; Claude AI targeted help)

my_train_file = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/repulsive/train.extxyz"
my_valid_file = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/repulsive/valid.extxyz"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/repulsive"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/repulsive"

my_atomic_numbers = "[1, 6, 7, 50, 53]" ##############

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_repprior",
    f"--train_file={my_train_file}",
    f"--valid_file={my_valid_file}",
    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=foundation", # try "isolated" after adding isolated atom energies to training set?
               #"mp" means from materials project (used to train foundation model)
    #"--model=MACE",
    #"--num_interactions=2", # keep default


    "--stress_key=stress",
    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=50",
    "--patience=10",
    "--batch_size=5",       # was 10
    "--valid_batch_size=5", # was 10
    "--device=cuda",
    "--default_dtype=float32", # less memory than 64

    "--r_max=5.0", #could be increased??
    "--pair_repulsion",           # Enables the built-in repulsive ZBL core prior
    "--distance_transform=Agnesi", # Smooths the transition between the core and the ML predictions


    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",

    # use a pre-trained ("foundation") model:
    "--foundation_model=small",           # Use MACE‑MP‑0 small
    "--multiheads_finetuning=True",        # Enable recommended Multihead Replay (for generalizable models; but expensive)
    "--pt_train_file=mp",                 # Required for Multihead Replay


    f"--atomic_numbers={my_atomic_numbers}", #[1, 6, 7, 50, 53]
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0", ##############
    "--forces_weight=100.0", #############

#OPTIMIZER
    "--optimizer=adamw",  #AdamW - optimizer
    "--amsgrad", #AMSGrad variant --> ensure stability
    "--weight_decay=5e-7",


#EMA/SWA
#optimization technique and loss function
#EMA: exponential smoothing
#SWA: stochastic weight averaging

#ema: Enables Exponential Moving Average during training.
#Instead of saving only the model weights from the current optimization step, it maintains a running average of the model parameters.
#This produces a smoother, more stable set of parameters, reducing variance and helping the model generalize better on unseen structures.
    "--ema",
    "--ema_decay=0.99",  #decay rate

#swa: Enables Stochastic Weight Averaging, a technique that averages model weights sampled across multiple late stage epochs
#find a broader, flatter local minimum in the loss landscape.
    "--swa",
    "--start_swa=30", #averaging will begin at epoch start_swa

#importance
    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",


#Dynamically normalizes the loss contributions based on the Root Mean Square (RMS) of the atomic forces in the dataset.
#This prevents large force magnitudes from dominating the gradients unevenly across different atomic configurations.
    "--scaling=rms_forces_scaling",


]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Simulation of the trajectory with ase

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/repulsive/model_3000_pts_repprior_run-1_stagetwo.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/repultive/md_trajectory_repulsive_2000.extxyz"

output_dir = os.path.dirname(output_traj_path)
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured output directory exists: {output_dir}")
# -------------------------------------------------------------------------------

atoms = read(init_structure_path, index=0)

calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

# Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

# Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=10)

# Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured output directory exists: /content/drive/MyDrive/MDProject/results/repultive
Loading initial structure...
Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/repulsive/model_3000_pts_repprior_run-1_stagetwo.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...


/tmp/ipykernel_1679/4165219618.py:43: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    0 | Temp:  301.1 K | Epot: -479.178 eV | Total E: -475.286 eV
Step:   10 | Temp:  241.5 K | Epot: -478.340 eV | Total E: -475.218 eV
Step:   20 | Temp:  247.8 K | Epot: -478.379 eV | Total E: -475.176 eV
Step:   30 | Temp:  265.5 K | Epot: -478.603 eV | Total E: -475.170 eV
Step:   40 | Temp:  283.7 K | Epot: -478.861 eV | Total E: -475.194 eV
Step:   50 | Temp:  260.7 K | Epot: -478.559 eV | Total E: -475.189 eV
Step:   60 | Temp:  283.4 K | Epot: -478.839 eV | Total E: -475.176 eV
Step:   70 | Temp:  295.9 K | Epot: -479.006 eV | Total E: -475.182 eV
Step:   80 | Temp:  314.3 K | Epot: -479.246 eV | Total E: -475.184 eV
Step:   90 | Temp:  324.4 K | Epot: -479.382 eV | Total E: -475.189 eV
Step:  100 | Temp:  296.2 K | Epot: -479.019 eV | Total E: -475.191 eV
Step:  110 | Temp:  300.4 K | Epot: -479.059 eV | Total E: -475.176 eV
Step:  120 | Temp:  301.5 K | Epot: -479.090 eV | Total E: -475.192 eV
Step:  130 | Temp:  341.8 K | Epot: -479.607 eV | Total E: -475.189 eV
Step: 

Analysis of the trajectory

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/repultive/md_trajectory_repulsive_2000.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/repulsivePotRes"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



[Total Energy] t0 = 6 (frame 6), g = 4.32, N_eff = 45.4 / 201
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8274
Area (180° peak): 0.1726
Max v dataset: 1.0356367599999992
Processed 201 frames successfully.
Processed 201 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 101128.35 K


In [ ]:
! pip install ase
! pip install pymbar
! pip install mace-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.4 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39523 sha256=a5b4940249b865516cf45d012369fd36fec132f621a6f4098ba649687626295a
  Stored in directory: /root/.cache/pip/wheels/05/2e/7e/39f980a5a0755f